<a href="https://colab.research.google.com/github/bh1rg1v/colab/blob/main/Torrent_To_Google_Drive_Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Code Starts Here

In [2]:
!python -m pip install --no-cache-dir libtorrent==2.0.15

import libtorrent as lt

print("Version:", lt.version)

ses = lt.session()
ses.listen_on(6881, 6891)

downloads = []

print("SUCCESS")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 19.7 MB/s  0:00:00
  Attempting uninstall: libtorrent
    Found existing installation: libtorrent 2.0.10
    Can't uninstall 'libtorrent'. No files were found to uninstall.
Version: 2.0.15.0
SUCCESS


/tmp/ipykernel_707/3020841344.py:8: DeprecationWarning: listen_on() is deprecated
  ses.listen_on(6881, 6891)


In [ ]:
from google.colab import files
import os

print("Upload your torrent file below:")
uploaded = files.upload()
torrent_path = list(uploaded.keys())[0]
print(f"Uploaded torrent: {torrent_path}")

Upload your torrent file below:


In [ ]:
import os
import libtorrent as lt

# Load torrent info
t_info = lt.torrent_info(torrent_path)

# Gather file info and sort by file name
files_list = []
for i in range(t_info.num_files()):
    f = t_info.file_at(i)
    files_list.append({
        'index': i,
        'path': f.path,
        'size': f.size
    })

# Sort files alphabetically based on path/name
files_sorted = sorted(files_list, key=lambda x: x['path'])

# Calculate batches (200GB = 200 * 1024 * 1024 * 1024 bytes)
BATCH_LIMIT = 200 * 1024 * 1024 * 1024
current_batch_size = 0
batches = []
current_batch = []

for file in files_sorted:
    if current_batch_size + file['size'] > BATCH_LIMIT and current_batch:
        batches.append(current_batch)
        current_batch = []
        current_batch_size = 0
    current_batch.append(file)
    current_batch_size += file['size']
if current_batch:
    batches.append(current_batch)

# Write details to /content/torrent_files_list.txt
output_path = "/content/torrent_files_list.txt"
with open(output_path, "w", encoding="utf-8") as f:
    f.write("=== ALL FILES (SORTED BY NAME) ===\n")
    for idx, item in enumerate(files_sorted):
        f.write(f"No: {idx} | Original Index: {item['index']} | Size: {item['size'] / (1024**2):.2f} MB | Name: {item['path']}\n")

    f.write("\n=== BATCHES (200GB LIMIT) ===\n")
    print("\n--- Torrent Batches (Sorted Alphabetically) ---")
    for b_idx, batch in enumerate(batches):
        start_file_num = files_sorted.index(batch[0])
        end_file_num = files_sorted.index(batch[-1])
        batch_size_gb = sum(item['size'] for item in batch) / (1024**3)
        batch_info = f"Batch {b_idx + 1}: Start Sorted No = {start_file_num}, End Sorted No = {end_file_num} (Total Size: {batch_size_gb:.2f} GB)"
        print(batch_info)
        f.write(batch_info + "\n")

print(f"\nSaved full list and batch layout to {output_path}")


--- Torrent Batches (Sorted Alphabetically) ---
Batch 1: Start Sorted No = 0, End Sorted No = 123 (Total Size: 199.72 GB)
Batch 2: Start Sorted No = 124, End Sorted No = 151 (Total Size: 193.12 GB)
Batch 3: Start Sorted No = 152, End Sorted No = 165 (Total Size: 185.12 GB)
Batch 4: Start Sorted No = 166, End Sorted No = 177 (Total Size: 196.10 GB)
Batch 5: Start Sorted No = 178, End Sorted No = 187 (Total Size: 194.98 GB)
Batch 6: Start Sorted No = 188, End Sorted No = 195 (Total Size: 180.21 GB)
Batch 7: Start Sorted No = 196, End Sorted No = 203 (Total Size: 195.87 GB)
Batch 8: Start Sorted No = 204, End Sorted No = 210 (Total Size: 187.76 GB)
Batch 9: Start Sorted No = 211, End Sorted No = 217 (Total Size: 198.26 GB)
Batch 10: Start Sorted No = 218, End Sorted No = 223 (Total Size: 181.40 GB)
Batch 11: Start Sorted No = 224, End Sorted No = 350 (Total Size: 198.45 GB)
Batch 12: Start Sorted No = 351, End Sorted No = 409 (Total Size: 198.50 GB)
Batch 13: Start Sorted No = 410, End S

/tmp/ipykernel_840/3228585513.py:10: DeprecationWarning: file_at() is deprecated
  f = t_info.file_at(i)
/tmp/ipykernel_840/3228585513.py:14: DeprecationWarning: file_entry is deprecated
  'size': f.size


In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
download_path = "/content/drive/My Drive/Torrent"
os.makedirs(download_path, exist_ok=True)

Mounted at /content/drive


In [ ]:
# Change these numbers to choose which sorted files to download
# Defaults to the first batch
start = 0
end = 123

# try:
#     # Automatically resolve default ending for the first batch if calculated
#     end = files_sorted.index(batches[0][-1])
# except:
#     pass

print(f"Configured sorted file range: Start = {start}, End = {end}")

Configured sorted file range: Start = 0, End = 123


In [ ]:
import sys
import time
import os

# Set download priorities: 1 for selected range, 0 for others
priorities = [0] * t_info.num_files()

selected_files = files_sorted[start:end+1]
for item in selected_files:
    priorities[item['index']] = 1

# Track the local destination of files being downloaded in this cell
downloaded_file_paths = []
for item in selected_files:
    full_dest_path = os.path.join(download_path, item['path'])
    downloaded_file_paths.append(full_dest_path)

# Write the paths of the downloaded files to a text file for later retrieval
recent_downloads_log = "/content/recently_downloaded.txt"
with open(recent_downloads_log, "w", encoding="utf-8") as f:
    for path in downloaded_file_paths:
        f.write(path + "\n")

# Add Torrent with file priorities
params = {
    "save_path": download_path,
    "ti": t_info,
    "file_priorities": priorities
}
handle = ses.add_torrent(params)
downloads.append(handle)

print("Downloading selected batch files...")
while not handle.is_seed():
    s = handle.status()
    print(f"\rProgress: {s.progress * 100:.2f}% | Down: {s.download_rate / 1000:.2f} kB/s | State: {s.state}", end="")
    sys.stdout.flush()
    time.sleep(2)
print(f"\nFinished downloading target range [{start} to {end}]. File list saved to {recent_downloads_log}")

Progress: 0.00% | Down: 0.00 kB/s | State: checking_resume_data

/tmp/ipykernel_840/3949642767.py:34: DeprecationWarning: is_seed() is deprecated
  while not handle.is_seed():


Progress: 9.55% | Down: 4739.64 kB/s | State: downloading

In [ ]:
import os
import shutil

# Load recently downloaded paths from the log file and delete them
recent_downloads_log = "/content/recently_downloaded.txt"

if os.path.exists(recent_downloads_log):
    with open(recent_downloads_log, "r", encoding="utf-8") as f:
        paths_to_delete = [line.strip() for line in f if line.strip()]

    print(f"Loaded {len(paths_to_delete)} paths to clean up from {recent_downloads_log}...")
    for path in paths_to_delete:
        if os.path.exists(path):
            try:
                if os.path.isdir(path):
                    shutil.rmtree(path)
                else:
                    os.remove(path)
                print(f"Successfully removed: {path}")
            except Exception as e:
                print(f"Failed to remove {path}: {e}")
        else:
            print(f"File or directory already absent: {path}")

    # Clear the log file content after deletion
    open(recent_downloads_log, 'w').close()
    print("Cleanup completed. Log file cleared.")
else:
    print(f"No log file found at {recent_downloads_log}. Nothing to delete.")